In [1]:
#importing the libraries
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df=pd.read_csv('data_for_predictions.csv')

In [3]:
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df.duplicated().any())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14606 entries, 0 to 14605
Data columns (total 64 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Unnamed: 0                                  14606 non-null  int64  
 1   id                                          14606 non-null  object 
 2   cons_12m                                    14606 non-null  float64
 3   cons_gas_12m                                14606 non-null  float64
 4   cons_last_month                             14606 non-null  float64
 5   forecast_cons_12m                           14606 non-null  float64
 6   forecast_discount_energy                    14606 non-null  float64
 7   forecast_meter_rent_12m                     14606 non-null  float64
 8   forecast_price_energy_off_peak              14606 non-null  float64
 9   forecast_price_energy_peak                  14606 non-null  float64
 10  forecast_p

In [4]:
print(df.head(5))

   Unnamed: 0                                id  cons_12m  cons_gas_12m  \
0           0  24011ae4ebbe3035111d65fa7c15bc57  0.000000      4.739944   
1           1  d29c2c54acc38ff3c0614d0a653813dd  3.668479      0.000000   
2           2  764c75f661154dac3a6c254cd082ea7d  2.736397      0.000000   
3           3  bba03439a292a1e166f80264c16191cb  3.200029      0.000000   
4           4  149d57cf92fc41cf94415803a877cb4b  3.646011      0.000000   

   cons_last_month  forecast_cons_12m  forecast_discount_energy  \
0         0.000000           0.000000                       0.0   
1         0.000000           2.280920                       0.0   
2         0.000000           1.689841                       0.0   
3         0.000000           2.382089                       0.0   
4         2.721811           2.650065                       0.0   

   forecast_meter_rent_12m  forecast_price_energy_off_peak  \
0                 0.444045                        0.114481   
1                 1.23

In [5]:
#The data is standardised
Y=df['churn']
X=df.drop(['id','churn'],axis=1)
print(Y.shape)
print(X.shape)

(14606,)
(14606, 62)


In [6]:
print(Y.value_counts())

churn
0    13187
1     1419
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import train_test_split
Xtrain,Xtest,Ytrain,Ytest=train_test_split(X,Y,test_size=0.2,random_state=42)

In [8]:
print(Xtrain.shape)
print(Ytrain.shape)

(11684, 62)
(11684,)


In [9]:
#There is severe class imbalance, so we need to deal with it
from imblearn.over_sampling import SMOTE
sm=SMOTE()
Xtrain_smote,Ytrain_smote=sm.fit_resample(Xtrain,Ytrain)

In [10]:
#making ml prediction models
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)
from sklearn.model_selection import cross_val_score

In [11]:
svc = SVC(
    kernel='rbf',
    probability=True,
    random_state=42
)

svc.fit(Xtrain_smote,Ytrain_smote)

Ypred_svc = svc.predict(Xtest)

In [12]:
print("Accuracy :",accuracy_score(Ytest,Ypred_svc))
print("Precision:",precision_score(Ytest,Ypred_svc))
print("Recall   :",recall_score(Ytest,Ypred_svc))
print("F1 Score :",f1_score(Ytest,Ypred_svc))
print("ROC AUC  :",roc_auc_score(Ytest,Ypred_svc))

Accuracy : 0.4787816563997262
Precision: 0.10143979057591623
Recall   : 0.5081967213114754
F1 Score : 0.1691216584833606
ROC AUC  : 0.4917750897348359


In [13]:
#cross validation score
cv_svc = cross_val_score(
    svc,
    Xtrain_smote,
    Ytrain_smote,
    cv=5,
    scoring='accuracy'
)

print("CV Scores:",cv_svc)
print("Mean CV:",cv_svc.mean())

CV Scores: [0.5243614  0.51064333 0.52317881 0.52530747 0.51017029]
Mean CV: 0.518732261116367


In [ ]:
#random forest model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf.fit(Xtrain_smote,Ytrain_smote)

Ypred_rf = rf.predict(Xtest)


print("Accuracy :",accuracy_score(Ytest,Ypred_rf))
print("Precision:",precision_score(Ytest,Ypred_rf))
print("Recall   :",recall_score(Ytest,Ypred_rf))
print("F1 Score :",f1_score(Ytest,Ypred_rf))
print("ROC AUC  :",roc_auc_score(Ytest,Ypred_rf))

cv_rf = cross_val_score(
    rf,
    Xtrain_smote,
    Ytrain_smote,
    cv=5,
    scoring='accuracy'
)

print("CV Scores:",cv_rf)
print("Mean CV:",cv_rf.mean())

Accuracy : 0.9017796030116358
Precision: 0.7368421052631579
Recall   : 0.09180327868852459
F1 Score : 0.16326530612244897
ROC AUC  : 0.5439910547053629
CV Scores: [0.74385052 0.9884106  0.99195837 0.99408704 0.99124882]
Mean CV: 0.941911069063387


In [24]:
rf_prob = rf.predict_proba(Xtest)[:,1]

roc_auc_score(Ytest, rf_prob)

0.645828974485865

In [15]:
!pip install xgboost

In [16]:
from xgboost import XGBClassifier
xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(Xtrain_smote,Ytrain_smote)

Ypred_xgb = xgb.predict(Xtest)

cv_xgb = cross_val_score(
    xgb,
    Xtrain_smote,
    Ytrain_smote,
    cv=5,
    scoring='accuracy'
)
print("Accuracy :",accuracy_score(Ytest,Ypred_xgb))
print("Precision:",precision_score(Ytest,Ypred_xgb))
print("Recall   :",recall_score(Ytest,Ypred_xgb))
print("F1 Score :",f1_score(Ytest,Ypred_xgb))
print("ROC AUC  :",roc_auc_score(Ytest,Ypred_xgb))
print("CV Scores:",cv_xgb)
print("Mean CV:",cv_xgb.mean())

Accuracy : 0.8963039014373717
Precision: 0.5208333333333334
Recall   : 0.08196721311475409
F1 Score : 0.141643059490085
ROC AUC  : 0.5365892618879082
CV Scores: [0.73817408 0.98675497 0.99030274 0.98982971 0.98888363]
Mean CV: 0.9387890255439924


In [20]:
#comparing all the models
from sklearn.metrics import classification_report
print('svc')
print(classification_report(Ytest,Ypred_svc))
print('rf')
print(classification_report(Ytest,Ypred_rf))
print('xgboost')
print(classification_report(Ytest,Ypred_xgb))

svc
              precision    recall  f1-score   support

           0       0.89      0.48      0.62      2617
           1       0.10      0.51      0.17       305

    accuracy                           0.48      2922
   macro avg       0.50      0.49      0.39      2922
weighted avg       0.81      0.48      0.57      2922

rf
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      2617
           1       0.74      0.09      0.16       305

    accuracy                           0.90      2922
   macro avg       0.82      0.54      0.56      2922
weighted avg       0.89      0.90      0.87      2922

xgboost
              precision    recall  f1-score   support

           0       0.90      0.99      0.94      2617
           1       0.52      0.08      0.14       305

    accuracy                           0.90      2922
   macro avg       0.71      0.54      0.54      2922
weighted avg       0.86      0.90      0.86      2922

